# Evaluador de consistencia cruzada — GPT-4o vs Gemini 2.5 Flash

Compara las extracciones estructuradas obtenidas por ambos modelos sobre una submuestra de control (15 imágenes, ~135 registros) para calcular el **Índice de Consistencia Global (ICG)**.

El evaluador no mide qué modelo transcribe mejor carácter a carácter, sino en qué medida ambos coinciden en los campos clave de la base de datos. Un alto grado de coincidencia entre modelos independientes aporta evidencia adicional sobre la fiabilidad de los registros sin necesidad de validación paleográfica experta exhaustiva.

El algoritmo trabaja en dos fases: emparejamiento de registros homólogos por nombre del bautizado y del padre (similitud difusa) y comparación campo a campo para clasificar cada par como COINCIDENCIA, CONFLICTO o HUÉRFANO.

> **Requisitos:** archivos Excel/CSV exportados por los notebooks de transcripción de OpenAI y Gemini · ver `REPRODUCIR.txt`.

## Instalación de dependencias

`rapidfuzz` acelera la comparación de cadenas. Si no está disponible, el notebook recurre automáticamente a `difflib`, incluida en la biblioteca estándar de Python.

In [ ]:
!pip install -q rapidfuzz openpyxl


## Montaje de Google Drive

Monta Google Drive para acceder a los archivos del proyecto.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Configuración

Valores de configuración básicos para reproducibilidad. No cambia la lógica del notebook.

In [ ]:
import os
DRIVE_MOUNT = '/content/drive'
DEFAULT_OPENAI = os.path.join(DRIVE_MOUNT, 'MyDrive', 'prueba_tfm', 'bautismos_ind_OPENAI.xlsx')
DEFAULT_GEMINI = os.path.join(DRIVE_MOUNT, 'MyDrive', 'prueba_tfm', 'bautismos_ind_GEMINI.xlsx')
print('Rutas por defecto:', DEFAULT_OPENAI, DEFAULT_GEMINI)

## Importaciones y función de similitud

Define dos funciones de similitud: `similitud` (estricta, para detectar conflictos entre registros ya emparejados) y `similitud_parcial` (tolerante a subcadenas, para el emparejamiento inicial donde un modelo puede haber volcado el nombre completo y el otro solo el nombre de pila).

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime

# Intentamos usar rapidfuzz (mas rapido); si no esta disponible, usamos
# difflib, que viene incluido en Python y no requiere instalacion.
try:
    from rapidfuzz import fuzz
    def similitud(a: str, b: str) -> float:
        """Similitud 'completa' (0-100) entre dos cadenas. Penaliza si una
        cadena es mucho mas larga que la otra, aunque una contenga a la otra.
        Se usa para detectar CONFLICTOS entre registros ya emparejados."""
        return fuzz.ratio(str(a).lower().strip(), str(b).lower().strip())

    def similitud_parcial(a: str, b: str) -> float:
        """Similitud 'parcial' (0-100): puntua alto si la cadena mas corta
        aparece practicamente entera dentro de la mas larga, aunque esta
        contenga texto adicional (p.ej. 'Gregorio' dentro de 'Gregoria de
        Florian Rubio y de Ynesma Cordero'). Se usa solo para EMPAREJAR
        registros entre modelos, donde un modelo puede haber volcado una
        frase completa en un campo que el otro dejo limpio."""
        return fuzz.partial_ratio(str(a).lower().strip(), str(b).lower().strip())

    print("Usando rapidfuzz para la comparacion de texto.")
except ImportError:
    from difflib import SequenceMatcher

    def similitud(a: str, b: str) -> float:
        """Similitud 'completa' (0-100) entre dos cadenas (fallback sin rapidfuzz).
        Se usa para detectar CONFLICTOS entre registros ya emparejados."""
        a, b = str(a).lower().strip(), str(b).lower().strip()
        return SequenceMatcher(None, a, b).ratio() * 100

    def similitud_parcial(a: str, b: str) -> float:
        """Similitud 'parcial' (0-100) (fallback sin rapidfuzz): compara la
        cadena mas corta contra la ventana de igual longitud, dentro de la
        cadena mas larga, que mejor encaje. Se usa solo para EMPAREJAR
        registros entre modelos."""
        a, b = str(a).lower().strip(), str(b).lower().strip()
        if not a or not b:
            return 0.0
        corta, larga = (a, b) if len(a) <= len(b) else (b, a)
        if len(corta) == 0:
            return 0.0
        mejor = 0.0
        # Si la cadena larga no es mucho mas larga, una comparacion directa basta
        if len(larga) <= len(corta) + 2:
            return SequenceMatcher(None, corta, larga).ratio() * 100
        # Deslizamos una ventana del tamano de la cadena corta sobre la larga
        paso = max(1, len(larga) // 40)  # limitar el numero de comparaciones
        for inicio in range(0, len(larga) - len(corta) + 1, paso):
            ventana = larga[inicio:inicio + len(corta)]
            score = SequenceMatcher(None, corta, ventana).ratio() * 100
            if score > mejor:
                mejor = score
        return mejor

    print("rapidfuzz no disponible: usando difflib como alternativa (similitud parcial aproximada).")


## Configuración

Define las columnas de identificación y los campos usados en cada fase del algoritmo.

- `campos_emparejamiento`: campos usados para decidir si dos registros de distinto modelo corresponden al mismo bautizo (`nombre_bautizado` y `nombre_padre`).
- `campos_comparacion`: todos los campos que se comparan una vez emparejado el registro para detectar coincidencias o conflictos.

In [ ]:
CONFIG = {
    # Columna que identifica la imagen de origen (ruta del archivo .jpg)
    "col_imagen": "archivo_imagen",

    # Columna con la posicion del acta dentro de la imagen (para el chequeo de huerfanos)
    "col_indice_acta": "indice_acta_en_imagen",

    # Campos clave para EMPAREJAR registros entre modelos
    "campos_emparejamiento": ["nombre_bautizado", "nombre_padre"],

    # Todos los campos que se comparan una vez emparejado el registro
    "campos_comparacion": [
        "nombre_bautizado",
        "fecha_bautizo",
        "fecha_nacimiento",
        "lugar_nacimiento_bautizado",
        "nombre_padre",
        "lugar_nacimiento_padre",
        "nombre_madre",
        "lugar_nacimiento_madre",
        "abuelo_paterno",
        "lugar_nacimiento_abuelo_paterno",
        "abuela_paterna",
        "lugar_nacimiento_abuela_paterna",
        "abuelo_materno",
        "lugar_nacimiento_abuelo_materno",
        "abuela_materna",
        "lugar_nacimiento_abuela_materna",
    ],

    # Columnas que no se comparan, pero se incluyen en el informe para dar contexto
    # a los casos de conflicto o revision manual
    "campos_informativos": ["transcripcion_acta", "notas_adicionales"],

    # Umbral de similitud (0-100) para considerar que dos registros son el mismo bautizo
    "umbral_emparejamiento": 75,

    # Umbral de similitud (0-100) para considerar un campo como coincidente
    "umbral_campo_coincidente": 85,

    # Campos donde, si la similitud EXACTA no llega al umbral, se admite
    # tambien la similitud PARCIAL (subcadena) como prueba de coincidencia.
    # Util para 'nombre_bautizado', que en algunas actas viene con la frase
    # completa del acta en un modelo y solo el nombre limpio en el otro.
    "campos_tolerantes_subcadena": ["nombre_bautizado"],

    # Margen de tolerancia para el indice de acta de un huerfano: se acepta
    # si su indice no supera (maximo de los registros emparejados + margen)
    "margen_indice_huerfano": 2,

    # Columna de fecha usada en el chequeo de coherencia de huerfanos
    "col_fecha": "fecha_bautizo",
}


## Carga de datos

Monta Drive y carga los archivos de extracción de ambos modelos. Ajusta las rutas a tus archivos reales antes de ejecutar.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
def cargar_datos(ruta_openai: str, ruta_gemini: str):
    """Carga las transcripciones de ambos modelos desde CSV o Excel."""
    def _cargar(ruta):
        if ruta.endswith(".xlsx") or ruta.endswith(".xls"):
            return pd.read_excel(ruta)
        return pd.read_csv(ruta)

    df_openai = _cargar(ruta_openai)
    df_gemini = _cargar(ruta_gemini)
    return df_openai, df_gemini


# --- AJUSTA ESTAS DOS RUTAS A TUS ARCHIVOS REALES ---
RUTA_OPENAI = "/content/drive/MyDrive/prueba_tfm/bautismos_ind_OPENAI.xlsx"
RUTA_GEMINI = "/content/drive/MyDrive/prueba_tfm/bautismos_ind_GEMINI.xlsx"

df_openai, df_gemini = cargar_datos(RUTA_OPENAI, RUTA_GEMINI)

print(f"OpenAI: {len(df_openai)} registros en {df_openai[CONFIG['col_imagen']].nunique()} imagenes")
print(f"Gemini: {len(df_gemini)} registros en {df_gemini[CONFIG['col_imagen']].nunique()} imagenes")
df_openai.head()


## Emparejamiento de registros

Para cada imagen, vincula los registros de OpenAI con los de Gemini usando un algoritmo greedy que maximiza la similitud sobre los campos de emparejamiento. Los registros sin pareja en el otro modelo quedan como huérfanos y se evalúan por separado.

In [ ]:
def calcular_similitud_registro(fila_a: pd.Series, fila_b: pd.Series, campos: list) -> float:
    """
    Similitud media entre dos registros, sobre la lista de campos dada.
    Usa similitud_parcial (no similitud exacta), porque en el emparejamiento
    nos interesa reconocer que 'Gregorio' y 'Gregoria de Florian Rubio y de
    Ynesma Cordero' probablemente sean el mismo bautizo, aunque un modelo
    haya volcado una frase completa en el campo y el otro lo haya dejado
    limpio. La similitud ESTRICTA (similitud, sin '_parcial') se reserva
    para detectar conflictos una vez el registro ya esta emparejado.
    """
    similitudes = [similitud_parcial(fila_a[c], fila_b[c]) for c in campos]
    return float(np.mean(similitudes))


def emparejar_registros_imagen(df_openai_img: pd.DataFrame, df_gemini_img: pd.DataFrame, config: dict):
    """
    Empareja los registros de una misma imagen entre OpenAI y Gemini.

    Usa un enfoque greedy: para cada registro de OpenAI, busca el registro
    de Gemini con mayor similitud (sobre campos_emparejamiento) que aun no
    haya sido emparejado. Si la similitud supera el umbral, se consideran
    el mismo registro.

    Devuelve:
        - lista de tuplas (idx_openai, idx_gemini) emparejadas
        - lista de indices de OpenAI sin pareja (huerfanos de OpenAI)
        - lista de indices de Gemini sin pareja (huerfanos de Gemini)
    """
    campos_emp = config["campos_emparejamiento"]
    umbral = config["umbral_emparejamiento"]

    indices_gemini_libres = list(df_gemini_img.index)
    emparejados = []

    for idx_o, fila_o in df_openai_img.iterrows():
        mejor_idx_g = None
        mejor_score = -1.0

        for idx_g in indices_gemini_libres:
            fila_g = df_gemini_img.loc[idx_g]
            score = calcular_similitud_registro(fila_o, fila_g, campos_emp)
            if score > mejor_score:
                mejor_score = score
                mejor_idx_g = idx_g

        if mejor_idx_g is not None and mejor_score >= umbral:
            emparejados.append((idx_o, mejor_idx_g))
            indices_gemini_libres.remove(mejor_idx_g)

    indices_openai_emparejados = {e[0] for e in emparejados}
    huerfanos_openai = [i for i in df_openai_img.index if i not in indices_openai_emparejados]
    huerfanos_gemini = indices_gemini_libres

    return emparejados, huerfanos_openai, huerfanos_gemini


## Clasificación de registros

Cada par emparejado se clasifica como:

- **COINCIDENCIA**: ambos modelos coinciden en los campos clave por encima del umbral de similitud.
- **CONFLICTO**: ambos modelos detectan el registro pero difieren en algún campo clave.
- **HUÉRFANO_ACEPTADO**: solo un modelo detecta el registro, pero supera los controles de coherencia interna (fecha válida, nombre no vacío, índice plausible).
- **HUÉRFANO_REVISION**: solo un modelo detecta el registro y no supera los controles; requiere revisión manual.

In [ ]:
def comparar_par_emparejado(fila_o: pd.Series, fila_g: pd.Series, config: dict) -> dict:
    """
    Compara campo a campo un par de registros ya emparejados.
    Devuelve un diccionario con el detalle de similitud por campo y si el
    registro en conjunto es COINCIDENCIA o CONFLICTO.

    Para los campos listados en 'campos_tolerantes_subcadena' (por defecto,
    nombre_bautizado), se usa ADEMAS la similitud_parcial: si esta supera
    el umbral, el campo se considera coincidente aunque la similitud exacta
    sea baja. Esto cubre el caso, observado en los datos reales, en que un
    modelo vuelca una frase completa ('Gregoria de Florian Rubio y de Ynesma
    Cordero') en un campo que el otro modelo dejo limpio ('Gregorio'): no es
    un error de contenido, es una diferencia de formato, y aqui se trata
    como tal en vez de marcarse como conflicto.
    """
    umbral_campo = config["umbral_campo_coincidente"]
    campos_tolerantes = config.get("campos_tolerantes_subcadena", [])
    detalle = {}
    hay_conflicto = False

    for campo in config["campos_comparacion"]:
        score = similitud(fila_o[campo], fila_g[campo])
        detalle[f"sim_{campo}"] = round(score, 1)

        campo_ok = score >= umbral_campo
        if not campo_ok and campo in campos_tolerantes:
            score_parcial = similitud_parcial(fila_o[campo], fila_g[campo])
            detalle[f"sim_{campo}_parcial"] = round(score_parcial, 1)
            if score_parcial >= umbral_campo:
                campo_ok = True

        if not campo_ok:
            hay_conflicto = True

    detalle["estado"] = "CONFLICTO" if hay_conflicto else "COINCIDENCIA"
    return detalle


def chequeo_coherencia_huerfano(fila: pd.Series, indice_maximo_esperado, config: dict) -> bool:
    """
    Chequeo de coherencia para aceptar un huerfano automaticamente:
    - La fecha de bautizo debe estar presente y ser una fecha valida.
    - El nombre del bautizado no debe estar vacio.
    - El indice_acta_en_imagen no debe exceder el numero de actas que,
      como maximo, ha detectado CUALQUIERA de los dos modelos en esa
      imagen (+1 de margen). Por ejemplo: si OpenAI detecto 7 actas y
      Gemini detecto 6, un huerfano de OpenAI con indice 7 es coherente
      (Gemini simplemente no lo detecto); pero un huerfano con indice 15
      en una imagen donde el maximo combinado es 7 resulta sospechoso,
      probablemente una alucinacion del modelo.
    """
    col_fecha = config["col_fecha"]
    col_indice = config["col_indice_acta"]

    valor_fecha = fila.get(col_fecha, None)
    if pd.isna(valor_fecha) or str(valor_fecha).strip() == "":
        return False
    try:
        pd.to_datetime(valor_fecha, dayfirst=True)
    except (ValueError, TypeError):
        return False

    if pd.isna(fila.get("nombre_bautizado")) or str(fila.get("nombre_bautizado")).strip() == "":
        return False

    indice_actual = fila.get(col_indice, None)
    if pd.notna(indice_actual) and indice_maximo_esperado is not None:
        margen = config.get("margen_indice_huerfano", 2)
        if indice_actual > indice_maximo_esperado + margen:
            return False

    return True


## Ejecución del proceso completo

Recorre todas las imágenes de la submuestra, empareja y clasifica los registros, y devuelve un DataFrame con el estado de cada uno.

In [ ]:
def ejecutar_validacion(df_openai: pd.DataFrame, df_gemini: pd.DataFrame, config: dict) -> pd.DataFrame:
    """
    Ejecuta el proceso de emparejamiento y clasificacion sobre todas las
    imagenes presentes en ambos dataframes.

    Devuelve un DataFrame con una fila por registro evaluado (de cualquiera
    de los dos modelos) con su estado final:
        COINCIDENCIA, CONFLICTO, HUERFANO_ACEPTADO, HUERFANO_REVISION
    """
    col_img = config["col_imagen"]
    col_indice = config["col_indice_acta"]
    campos_info = config.get("campos_informativos", [])
    imagenes = sorted(set(df_openai[col_img]) | set(df_gemini[col_img]))

    resultados = []

    for imagen_id in imagenes:
        df_o_img = df_openai[df_openai[col_img] == imagen_id]
        df_g_img = df_gemini[df_gemini[col_img] == imagen_id]

        emparejados, huerfanos_o, huerfanos_g = emparejar_registros_imagen(df_o_img, df_g_img, config)

        # Indice de referencia: el maximo indice de acta entre los registros
        # YA EMPAREJADOS (confirmados por ambos modelos). Se usa para juzgar
        # si un huerfano cae dentro de un rango plausible para esta imagen,
        # en lugar de usar el maximo en bruto de cada modelo por separado
        # (que incluiria al propio huerfano sospechoso en su propio calculo).
        indices_emparejados = []
        for idx_o, idx_g in emparejados:
            valor = df_o_img.loc[idx_o, col_indice]
            if pd.notna(valor):
                indices_emparejados.append(valor)
        indice_maximo_esperado = max(indices_emparejados) if indices_emparejados else None

        # --- Pares emparejados: coincidencia o conflicto ---
        for idx_o, idx_g in emparejados:
            fila_o = df_o_img.loc[idx_o]
            fila_g = df_g_img.loc[idx_g]
            detalle = comparar_par_emparejado(fila_o, fila_g, config)

            fila_resultado = {
                "imagen_id": imagen_id,
                "origen": "OpenAI+Gemini",
                "nombre_bautizado_openai": fila_o["nombre_bautizado"],
                "nombre_bautizado_gemini": fila_g["nombre_bautizado"],
                **detalle,
            }
            for campo in campos_info:
                fila_resultado[f"{campo}_openai"] = fila_o.get(campo)
                fila_resultado[f"{campo}_gemini"] = fila_g.get(campo)
            resultados.append(fila_resultado)

        # --- Huerfanos de OpenAI ---
        for idx_o in huerfanos_o:
            fila_o = df_o_img.loc[idx_o]
            aceptado = chequeo_coherencia_huerfano(fila_o, indice_maximo_esperado, config)
            fila_resultado = {
                "imagen_id": imagen_id,
                "origen": "Solo OpenAI",
                "nombre_bautizado_openai": fila_o["nombre_bautizado"],
                "nombre_bautizado_gemini": None,
                "estado": "HUERFANO_ACEPTADO" if aceptado else "HUERFANO_REVISION",
            }
            for campo in campos_info:
                fila_resultado[f"{campo}_openai"] = fila_o.get(campo)
                fila_resultado[f"{campo}_gemini"] = None
            resultados.append(fila_resultado)

        # --- Huerfanos de Gemini ---
        for idx_g in huerfanos_g:
            fila_g = df_g_img.loc[idx_g]
            aceptado = chequeo_coherencia_huerfano(fila_g, indice_maximo_esperado, config)
            fila_resultado = {
                "imagen_id": imagen_id,
                "origen": "Solo Gemini",
                "nombre_bautizado_openai": None,
                "nombre_bautizado_gemini": fila_g["nombre_bautizado"],
                "estado": "HUERFANO_ACEPTADO" if aceptado else "HUERFANO_REVISION",
            }
            for campo in campos_info:
                fila_resultado[f"{campo}_openai"] = None
                fila_resultado[f"{campo}_gemini"] = fila_g.get(campo)
            resultados.append(fila_resultado)

    return pd.DataFrame(resultados)


print("Ejecutando emparejamiento y validacion cruzada...")
df_resultados = ejecutar_validacion(df_openai, df_gemini, CONFIG)
print(f"Total de registros evaluados: {len(df_resultados)}")
df_resultados.head(10)


## Índice de Consistencia Global (ICG)

Calcula el ICG como el porcentaje de registros válidos sin necesidad de revisión manual (COINCIDENCIA + HUÉRFANO_ACEPTADO) sobre el total de registros únicos detectados por al menos uno de los dos modelos.

In [ ]:
def calcular_indice_consistencia(df_resultados: pd.DataFrame) -> dict:
    """
    Calcula el Indice de Consistencia Global (ICG):

        ICG = (COINCIDENCIA + HUERFANO_ACEPTADO) / total de registros evaluados

    Es decir, el porcentaje de registros que se consideran validos sin
    necesidad de revision manual, sobre el total de registros unicos
    detectados por al menos uno de los dos modelos.
    """
    total = len(df_resultados)
    conteo = df_resultados["estado"].value_counts().to_dict()

    coincidencias = conteo.get("COINCIDENCIA", 0)
    conflictos = conteo.get("CONFLICTO", 0)
    huerfanos_aceptados = conteo.get("HUERFANO_ACEPTADO", 0)
    huerfanos_revision = conteo.get("HUERFANO_REVISION", 0)

    validos_sin_revision = coincidencias + huerfanos_aceptados
    icg = round(100 * validos_sin_revision / total, 2) if total > 0 else 0.0

    return {
        "total_registros_evaluados": total,
        "coincidencias": coincidencias,
        "conflictos": conflictos,
        "huerfanos_aceptados": huerfanos_aceptados,
        "huerfanos_revision_manual": huerfanos_revision,
        "indice_consistencia_global_%": icg,
    }


resumen = calcular_indice_consistencia(df_resultados)

print("--- RESUMEN ---")
for clave, valor in resumen.items():
    print(f"  {clave}: {valor}")

# Aviso de diagnostico: si no hay NINGUN registro emparejado (coincidencias +
# conflictos = 0) pero si hay huerfanos, casi seguro que el emparejamiento esta
# fallando por algun motivo estructural (rutas de archivo_imagen distintas
# entre OpenAI y Gemini, o un campo de emparejamiento con formato muy distinto
# entre ambos modelos), no porque los modelos no compartan ningun bautizo real.
if resumen["coincidencias"] + resumen["conflictos"] == 0 and resumen["total_registros_evaluados"] > 0:
    print()
    print("AVISO: no se ha emparejado NINGUN registro entre OpenAI y Gemini.")
    print("Esto suele indicar un problema de emparejamiento, no de los datos en si.")
    print("Revisa: (1) que 'archivo_imagen' tenga el MISMO valor en ambos ficheros")
    print("para la misma acta, y (2) que 'campos_emparejamiento' en CONFIG sean")
    print("razonables para tus datos.")


## Exportación del informe

Genera un Excel con dos pestañas: **Resumen** (ICG y conteos) y **Detalle** (registro por registro, ordenado para que los casos que requieren revisión aparezcan primero).

In [ ]:
def generar_informe(df_resultados: pd.DataFrame, resumen: dict, ruta_salida: str):
    """
    Exporta un Excel con dos hojas:
        - 'Resumen': el indice de consistencia global y los conteos.
        - 'Detalle': el detalle fila a fila, ordenado para que los casos
          que requieren revision manual (CONFLICTO y HUERFANO_REVISION)
          aparezcan primero.
    """
    orden_prioridad = {"CONFLICTO": 0, "HUERFANO_REVISION": 1, "HUERFANO_ACEPTADO": 2, "COINCIDENCIA": 3}
    df_ordenado = df_resultados.copy()
    df_ordenado["_orden"] = df_ordenado["estado"].map(orden_prioridad)
    df_ordenado = df_ordenado.sort_values(["_orden", "imagen_id"]).drop(columns="_orden")

    df_resumen = pd.DataFrame([resumen])

    with pd.ExcelWriter(ruta_salida, engine="openpyxl") as writer:
        df_resumen.to_excel(writer, sheet_name="Resumen", index=False)
        df_ordenado.to_excel(writer, sheet_name="Detalle", index=False)

    print(f"Informe exportado a: {ruta_salida}")


RUTA_INFORME = "informe_consistencia.xlsx"
generar_informe(df_resultados, resumen, RUTA_INFORME)

print(f"\nFecha de ejecucion: {datetime.now().strftime('%Y-%m-%d %H:%M')}")


## Descarga del informe

Descarga el Excel al ordenador si se ejecuta en Google Colab. Fuera de Colab, el archivo ya está guardado en el directorio de trabajo.

In [ ]:
try:
    from google.colab import files
    files.download(RUTA_INFORME)
except ImportError:
    print("No estas en Google Colab: el archivo ya esta guardado en el directorio de trabajo.")
